# Projet fin de module – Partie II : Données textuelles
## Dataset : 20 Newsgroups | Pipeline textuel scikit-learn
Auteur : [Nom Étudiant] | EMSI 2025-2026

### Section 1 : Chargement et présentation du corpus

Nous travaillons sur une classification supervisée de textes issus de 4 groupes Usenet. L'objectif est d'implémenter des pipelines associant extraction de caractéristiques textuelles (Vectorizers) et classifieurs (Naive Bayes et Linear SVM).

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sélection de 4 catégories distinctes
categories = ['sci.space', 'rec.sport.hockey', 'talk.politics.guns', 'comp.graphics']

# Chargement des données en enlevant les éléments d'en-tête, de pied de page et les citations pour éviter le leakage
train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'), random_state=42)
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers', 'footers', 'quotes'), random_state=42)

print(f"Train : {len(train_data.data)} documents | {len(train_data.target_names)} classes")
print(f"Test  : {len(test_data.data)} documents")

### Description du corpus et difficultés anticipées

Le dataset 20 Newsgroups présente plusieurs défis majeurs :
1. **Bruit textuel important** : Présence de fautes de frappe, d'abréviations, d'argot Usenet. L'option `remove=('headers', 'footers', 'quotes')` permet d'écarter les signatures et les en-têtes qui contiennent souvent le nom du groupe (ce qui fausserait l'évaluation).
2. **Matrices de vocabulaire très creuses (sparse)** : Avec des milliers de documents, le vocabulaire total contient des dizaines de milliers de mots uniques. Chaque document n'utilisant qu'une fraction de ces mots, la représentation vectorielle contiendra plus de 98% de zéros. L'utilisation de formats de stockage creux (CSR matrix) est indispensable pour limiter l'empreinte mémoire.
3. **Ambiguïté inter-classes** : Certains termes peuvent apparaître dans plusieurs contextes (par exemple, le mot 'orbit' ou 'space' peut être utilisé au sens figuré dans la politique ou l'informatique graphique).

In [ ]:
# Distribution des classes
plt.figure(figsize=(8, 4))
sns.countplot(x=train_data.target, palette='viridis')
plt.xticks(ticks=range(4), labels=train_data.target_names, rotation=15)
plt.title("Distribution des documents par classe (Train Set)")
plt.xlabel("Catégorie")
plt.ylabel("Nombre de documents")
plt.show()

# Distribution des longueurs de documents
doc_lengths = [len(doc.split()) for doc in train_data.data]
plt.figure(figsize=(10, 4))
sns.histplot(doc_lengths, bins=50, kde=True, color='purple', log_scale=True)
plt.title("Distribution des longueurs des documents (nombre de mots)")
plt.xlabel("Longueur du document (Log Scale)")
plt.ylabel("Nombre de documents")
plt.show()

### Section 2 : Représentations textuelles comparées

#### Théorie des vectoriseurs
- **CountVectorizer (Bag of Words)** : Représente chaque document par les fréquences brutes des mots qu'il contient. Limite : les mots courants (stop words comme 'the', 'is') dominent les vecteurs au détriment des mots porteurs de sens.
- **TF-IDF (Term Frequency - Inverse Document Frequency)** : Pondère le score de chaque mot. Le terme est valorisé s'il apparaît souvent dans un document donné, mais pénalisé s'il apparaît dans une trop grande proportion des documents du corpus. Formule : \(TF\text{-}IDF(t,d) = tf(t,d) \times \log(1 + N/df(t))\).
- **N-grammes** : Associe des séquences de mots adjacents (ex: bigrammes). Utile pour capter des expressions composées comme 'machine learning', au prix d'une explosion de la dimension.
- **HashingVectorizer** : Utilise le principe du hashing pour projeter les mots dans un espace vectoriel de dimension fixe sans stocker de dictionnaire en mémoire. Utile sur de très grands volumes de données.

In [ ]:
# Initialisation des représentations
vectorizers = {
    'CountVectorizer': CountVectorizer(max_features=10000, stop_words='english'),
    'TF-IDF Unigrams': TfidfVectorizer(max_features=10000, stop_words='english'),
    'TF-IDF Bigrams':  TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2)),
    'HashingVectorizer': HashingVectorizer(n_features=2**16, stop_words='english', alternate_sign=False)
}

results = {}
for name, vect in vectorizers.items():
    X_tr = vect.fit_transform(train_data.data)
    X_te = vect.transform(test_data.data)
    clf = MultinomialNB(alpha=1.0)
    clf.fit(X_tr, train_data.target)
    acc = clf.score(X_te, test_data.target)
    results[name] = acc
    print(f"{name:20s} -> Accuracy : {acc:.4f} | Sparse matrix shape : {X_tr.shape}")

# Graphique comparatif
plt.figure(figsize=(9, 5))
sns.barplot(x=list(results.keys()), y=list(results.values()), palette='Blues_d', edgecolor='black')
plt.ylim(0.85, 0.92)
plt.ylabel("Accuracy sur le Test Set")
plt.title("Performance de classification selon la représentation vectorielle (Naive Bayes)")
plt.grid(axis='y', alpha=0.3)
plt.show()

**Interprétation de la comparaison des représentations :**
La représentation TF-IDF avec Unigrammes fournit la meilleure performance de classification avec une exactitude de **89.78%** sur l'ensemble de test, suivie de près par le CountVectorizer (89.39%) et le TF-IDF avec Bigrammes (89.33%). Le HashingVectorizer obtient 88.87%. TF-IDF se montre supérieur en diminuant l'importance des termes redondants à l'échelle du corpus.

### Section 3 : Pipelines de classification textuelle

Nous mettons en place deux pipelines complets :
1. **Pipeline 1 (Baseline)** : TF-IDF Unigrams + Naive Bayes
2. **Pipeline 2** : TF-IDF Bigrams + SVM Linéaire (implémenté via `SGDClassifier(loss='hinge')`)

In [ ]:
# Pipeline 1 : TF-IDF + Naive Bayes
pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('clf', MultinomialNB(alpha=1.0))
])
pipeline_nb.fit(train_data.data, train_data.target)
y_pred_nb = pipeline_nb.predict(test_data.data)

# Pipeline 2 : TF-IDF Bigrams + SVM Linéaire
pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=15000, stop_words='english', ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', SGDClassifier(loss='hinge', alpha=1e-4, max_iter=100, random_state=42))
])
pipeline_svm.fit(train_data.data, train_data.target)
y_pred_svm = pipeline_svm.predict(test_data.data)

print("=== CLASSIFICATION REPORT : NAIVE BAYES ===")
print(classification_report(test_data.target, y_pred_nb, target_names=train_data.target_names))

print("\n=== CLASSIFICATION REPORT : SVM LINEAIRE ===")
print(classification_report(test_data.target, y_pred_svm, target_names=train_data.target_names))

#### Justification théorique des modèles
- **Naive Bayes** : Repose sur l'application du théorème de Bayes avec l'hypothèse de l'indépendance conditionnelle des variables (les mots apparaissent indépendamment les uns des autres dans une classe). Malgré cette simplification forte, il est extrêmement performant en text mining, très rapide à entraîner et nécessite peu de données.
- **SVM Linéaire (SGDClassifier)** : Modélise une frontière de décision linéaire de marge maximale séparant les classes dans l'espace de haute dimension. Particulièrement performant lorsque le nombre de caractéristiques (mots/n-grammes) est très grand.

### Section 4 : Évaluation complète

#### 4.1 Matrices de confusion

Les matrices de confusion permettent d'identifier les paires de classes les plus sujettes aux erreurs de classification.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ConfusionMatrixDisplay.from_predictions(
    test_data.target, y_pred_nb, 
    display_labels=train_data.target_names, ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title("Matrice de confusion – Naive Bayes")

ConfusionMatrixDisplay.from_predictions(
    test_data.target, y_pred_svm, 
    display_labels=train_data.target_names, ax=axes[1], cmap='Purples', colorbar=False
)
axes[1].set_title("Matrice de confusion – SVM")

plt.tight_layout()
plt.show()

**Interprétation des matrices de confusion :**
On observe que pour Naive Bayes, la catégorie la plus difficile à classifier est `talk.politics.guns` avec 68 documents mal classés (dont 39 sont classés comme `sci.space`). Pour le modèle SVM, le nombre de faux négatifs pour `talk.politics.guns` est de 53, montrant que les deux modèles partagent des difficultés similaires sur cette classe politique qui contient parfois du vocabulaire commun (par exemple des mots techniques ou sociétaux généraux).

#### 4.2 Courbes d'apprentissage (Learning Curves)

Elles permettent d'évaluer le comportement du modèle face à l'augmentation de la taille des données d'entraînement (underfitting vs overfitting).

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    pipeline_svm, train_data.data, train_data.target,
    train_sizes=np.linspace(0.1, 1.0, 8), cv=5, scoring='accuracy', n_jobs=-1
)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Exactitude Entraînement', color='steelblue')
plt.plot(train_sizes, val_scores.mean(axis=1), 's-', label='Exactitude Validation (CV)', color='coral')
plt.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                              train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='steelblue')
plt.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                              val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1, color='coral')
plt.xlabel("Taille de l'ensemble d'entraînement")
plt.ylabel("Accuracy")
plt.title("Courbe d'apprentissage – Pipeline TF-IDF + SVM")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Interprétation des courbes d'apprentissage :**
L'exactitude sur le jeu d'entraînement démarre proche de 100% et diminue légèrement à mesure que le corpus grandit et se diversifie. À l'inverse, l'exactitude de validation croisée augmente continuellement pour atteindre un plateau à ~88%. L'écart entre les courbes suggère une légère tendance à l'overfitting typique des modèles SVM linéaires sur de petits corpus de texte, mais le modèle se comporte de manière saine.

#### 4.3 Analyse des descripteurs les plus discriminants par classe (Naive Bayes)

In [ ]:
tfidf_vec = pipeline_nb['tfidf']
nb_clf = pipeline_nb['clf']
feature_names = np.array(tfidf_vec.get_feature_names_out())

print("=== TOP 15 CARACTERISTIQUES LES PLUS DISCRIMINANTES PAR CLASSE ===")
for i, class_name in enumerate(train_data.target_names):
    top_indices = np.argsort(nb_clf.feature_log_prob_[i])[-15:][::-1]
    top_features = feature_names[top_indices]
    print(f"\nClasse : {class_name.upper()}")
    print(", ".join(top_features))

**Interprétation des descripteurs discriminants :**
Les mots clés ressortant de chaque classe correspondent bien au domaine thématique :
- `comp.graphics` : mots techniques du domaine (`graphics`, `image`, `file`, `program`...)
- `rec.sport.hockey` : mots du sport (`game`, `team`, `hockey`, `nhl`...)
- `sci.space` : mots de l'astronomie (`space`, `nasa`, `orbit`, `launch`...)
- `talk.politics.guns` : mots de la politique des armes (`gun`, `people`, `weapons`, `government`...)

Cela valide l'adéquation de la représentation vectorielle et la cohérence de l'apprentissage.

#### 4.4 Analyse des erreurs de classification (SVM)

In [ ]:
misclassified_idx = np.where(y_pred_svm != test_data.target)[0]
print(f"Nombre de documents mal classés par le SVM : {len(misclassified_idx)}")
print("\n=== ECHANTILLON DE 5 DOCUMENTS MAL CLASSES ===\n")
for idx in misclassified_idx[:5]:
    true_label = train_data.target_names[test_data.target[idx]]
    pred_label = train_data.target_names[y_pred_svm[idx]]
    doc_snippet = test_data.data[idx].strip().replace('\n', ' ')[:150]
    print(f"[Index {idx}]")
    print(f"  Véritable Classe : {true_label}")
    print(f"  Classe Prédite   : {pred_label}")
    print(f"  Extrait Document : {doc_snippet}...")
    print("-" * 80)

#### Conclusion sur le compromis de classification textuelle

Sur ce dataset de 4 catégories de 20 Newsgroups, le classifieur **Naive Bayes** avec TF-IDF Unigrams montre une exactitude de **89.78%**, surpassant légèrement le **SVM** (87.97%). Ce résultat montre que Naive Bayes reste un choix robuste lorsque les classes sont bien définies par des mots clés spécifiques (ce qui est typiquement le cas pour nos 4 classes Usenet).

Des pistes d'amélioration pour accroître la performance incluent :
- Un nettoyage du texte plus poussé (lemmatisation ou stemmatisation avec Spacy/NLTK pour regrouper les formes grammaticales d'un même mot).
- L'utilisation de plongements lexicaux denses (Word2Vec, FastText) ou de modèles de transformeurs pré-entraînés (BERT, RoBERTa) qui capturent la sémantique profonde et le contexte des phrases, au détriment d'une complexité de calcul nettement accrue.